In [2]:
# intersperses communcation(multiattention) and the computation(ffwd)
import torch
from torch import nn
from torch.nn import functional as F

#hyperparameters
batch_size = 32
block_size = 8
max_iter = 5000
eval_interval = 300
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iter = 200
n_embd = 32
# ---------------------

torch.manual_seed(1337)

with open('../data/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
    
chars = sorted(list(set(text)))
vocab_size = len(chars)

#map functions
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[ch] for ch in s] #take a string and map to list of ints
decode = lambda l: "".join([itos[i] for i in l]) #take a list of ints and map to string

#split 
data = torch.tensor(encode(text), dtype = torch.long)
n = int(0.9 * len(data))
train = data[:n]
val   = data[n:]

#data load 
def get_batch(split:str):
    data = train if split == 'train' else val
    ix = torch.randint(len(data) - block_size, (batch_size, ))
    x = torch.stack([data[i : i + block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iter)
        for k in range(eval_iter):
            x, y = get_batch(split)
            logits, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out 

class Head(nn.Module):
    "one head of self-attention"
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size))) #tril is not a parameter, put in buffer
    
    def forward(self, x):
        B, T, C = x.shape 
        k = self.key(x)     #BT C = head_size
        q = self.query(x)   #BT C = head_size
        
        #compute attention "affinities"
        wei = q @ k.transpose(-2, -1)  * C**-0.5  #BTT
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
                
        v = self.value(x)   #BT C = head_size
        out = wei @ v #BTT @  #BT C ->  #BT C = head_size
        
        return out 

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        
    def forward(self, x):
        # x BT C = n_embd -> h(x) BT C = head_size
        return torch.cat([h(x) for h in self.heads], dim=-1)        

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),            
            nn.Linear(4 * n_embd, n_embd)
        )
    
    def forward(self, x):
        return self.net(x)
        


In [3]:
class Block(nn.Module):
    """ Transformer block: communication followed by computation """
    
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        
    def forward(self, x):
        x = self.sa(x)
        x = self.ffwd(x)
        return x 
        

In [4]:
        
class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd) #number of embedding dimensions
        self.position_embedding_table = nn.Embedding(block_size, n_embd) # each position has its own embd vec
        
        self.blocks = nn.Sequential(
            Block(n_embd, n_head=4), #32//4 = 8 4 attention with 8-dim then computation through ffwd
            Block(n_embd, n_head=4),
            Block(n_embd, n_head=4),            
        )
        
        self.lm_head = nn.Linear(n_embd, vocab_size)
        
    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx) #(BTC) C = n_embd
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) #(TC)
        x = tok_emb + pos_emb #now the token holds token identities and position it occur                

        x = self.blocks(x) #BT C = n_embd
    
        logits = self.lm_head(x) #BTC C = vocab_size
        
        
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        #idx is BT
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:] #BT = 8, since block size= 8 in the model, if don trunk the self(idx) cannot work
            logits, loss = self(idx_cond) #BTC
            logits = logits[:, -1, :] #BC
            probs = F.softmax(logits, dim=-1) #BC
            idx_next = torch.multinomial(probs, num_samples = 1) #B1
            idx = torch.cat([idx, idx_next], dim=1) #B T+1
        return idx
    

In [5]:
model = BigramLanguageModel()    
m = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

for iter in range(max_iter):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        
    xb, yb = get_batch('train')        
    
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    

step 0: train loss 4.1454, val loss 4.1450
step 300: train loss 3.1458, val loss 3.1623
step 600: train loss 2.7878, val loss 2.7811
step 900: train loss 2.6218, val loss 2.6205
step 1200: train loss 2.5527, val loss 2.5446
step 1500: train loss 2.5209, val loss 2.5120
step 1800: train loss 2.4476, val loss 2.4610
step 2100: train loss 2.4206, val loss 2.4326
step 2400: train loss 2.3978, val loss 2.4112
step 2700: train loss 2.3751, val loss 2.3959
step 3000: train loss 2.3554, val loss 2.3585
step 3300: train loss 2.3379, val loss 2.3399
step 3600: train loss 2.3043, val loss 2.3264
step 3900: train loss 2.2820, val loss 2.3048
step 4200: train loss 2.2862, val loss 2.2955
step 4500: train loss 2.2448, val loss 2.2826
step 4800: train loss 2.2410, val loss 2.2590


In [ ]:
#acually, when the nn become very deep, the performence actually not good as jut one layer of ffwd
# 优化困难(网络太深传不动梯度)。 vanishing gradient

In [9]:
context = torch.zeros((1,1), dtype = torch.long, device = device)
print(decode(m.generate(context, 500)[0].tolist()))


Tse now, at in ore of thap''s-dtpele gumine the, bees
I'd hightale in crest ver;
The 'this whan dIcungeSe af his arcodaregaved?
Thy bund ay, shackone
Tin pert shat set,
And gorl; ge nof sous, I thy toe love.

IVI:
The
Gcargees seang hy herge, do wadt wargede wn Rmall thees he prope co'r oves; notughtooo has
Hheade! Mordule?

Mund ty!

Whe demangh souor; bues'ds,
Ant
to it on a, gore triove spen: is and hasts pyon proin rathiRnddeiriet will
Ffe the de bose, foistiad;

Whis eur the goure,
Mardch, 


In [ ]:
# Residual Connection
""" 解决方案 1:Residual Connection(残差连接)★ 最关键

  改成"加"而不是"替换":

  def forward(self, x):
      x = x + self.sa(x)      # ← 加 x +
      x = x + self.ffwd(x)    # ← 加 x +
      return x

  为什么这一个 x + 就救活了深层网络:

  - 原来:x = sa(x),x 被完全替换。
  - 现在:x = x + sa(x),sa 只需学"在原基础上加什么改动"(残差 residual),原始信息 x
  通过 + 直接流过去。
  - 梯度高速公路:反向传播时,梯度能顺着这个 + 直接流回浅层(加法的梯度是
  1,不衰减),不用硬穿过每一层。深层网络瞬间好训练。

  这是 ResNet 的核心思想,也是 Transformer
  能叠几十上百层的关键。这个是你现在最缺、最该加的。 """

In [10]:
class Block(nn.Module):
    """ Transformer block: communication followed by computation """
    
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        
    def forward(self, x):
        x = x + self.sa(x)
        x = x + self.ffwd(x)
        return x 
        

In [11]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        
    def forward(self, x):
        # x BT C = n_embd -> h(x) BT C = head_size
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out 

In [12]:
        
class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd) #number of embedding dimensions
        self.position_embedding_table = nn.Embedding(block_size, n_embd) # each position has its own embd vec
        
        self.blocks = nn.Sequential(
            Block(n_embd, n_head=4), #32//4 = 8 4 attention with 8-dim then computation through ffwd
            Block(n_embd, n_head=4),
            Block(n_embd, n_head=4),            
        )
        
        self.lm_head = nn.Linear(n_embd, vocab_size)
        
    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx) #(BTC) C = n_embd
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) #(TC)
        x = tok_emb + pos_emb #now the token holds token identities and position it occur                

        x = self.blocks(x) #BT C = n_embd
    
        logits = self.lm_head(x) #BTC C = vocab_size
        
        
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        #idx is BT
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:] #BT = 8, since block size= 8 in the model, if don trunk the self(idx) cannot work
            logits, loss = self(idx_cond) #BTC
            logits = logits[:, -1, :] #BC
            probs = F.softmax(logits, dim=-1) #BC
            idx_next = torch.multinomial(probs, num_samples = 1) #B1
            idx = torch.cat([idx, idx_next], dim=1) #B T+1
        return idx
    

In [13]:
model = BigramLanguageModel()    
m = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

for iter in range(max_iter):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        
    xb, yb = get_batch('train')        
    
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    

step 0: train loss 4.7843, val loss 4.7673
step 300: train loss 2.5174, val loss 2.5221
step 600: train loss 2.3642, val loss 2.3707
step 900: train loss 2.2995, val loss 2.3063
step 1200: train loss 2.2249, val loss 2.2486
step 1500: train loss 2.2006, val loss 2.2390
step 1800: train loss 2.1763, val loss 2.2073
step 2100: train loss 2.1446, val loss 2.1846
step 2400: train loss 2.1293, val loss 2.1750
step 2700: train loss 2.1058, val loss 2.1741
step 3000: train loss 2.0911, val loss 2.1506
step 3300: train loss 2.0759, val loss 2.1445
step 3600: train loss 2.0550, val loss 2.1247
step 3900: train loss 2.0470, val loss 2.1241
step 4200: train loss 2.0329, val loss 2.1140
step 4500: train loss 2.0243, val loss 2.1187
step 4800: train loss 2.0180, val loss 2.0824


In [20]:
context = torch.zeros((1,1), dtype = torch.long, device = device)
print(decode(m.generate(context, 500)[0].tolist()))


There moust.

Ferly I to vente
you chair;
Andst I mane.
A to carfion, Clens;
For peeed
Pins to aa suctill sak it whice sile'st
So a spiglemeq his my marks dought osf broses:
His a tthought, mighter,-,
O in Loods is colterf, throuse, ventain; as Antle my we a not honoumpsed 
Leen.
God Yeleing ereat, choll give pown that Ye his to bearfor Yons?

HENNMPEO:
Dakidet see.
Contes of shall, Cact,
I sence
Have rint PAREW:
Beast, so made tafe!

CTIUS:
Meebale say is chion.

KING ELASULIESTER:
Go bots.

GR


In [ ]:
""" 好,residual
  connection(残差连接)是深度学习里最重要的想法之一,我从头讲透,不用你有背景。

  一句话先建立直觉

  ▎ 不要让每一层"从头重写"输入,而是让它只学"在输入基础上做什么微调",原始输入直接加
  ▎ 过去。

  代码上就一个字的差别:

  x = layer(x)        # ❌ 替换:x 被彻底改写
  x = x + layer(x)    # ✅ 残差:保留原 x,只加上 layer 学到的"改动"

  那个 x + 就是 residual connection。看着简单,但它解决了一个致命问题。

  先讲"没有残差"会出什么事

  想象一条深网络,数据一层层往下传:

  x → layer1 → layer2 → layer3 → ... → layer10 → 输出

  每层都把 x 完全替换成新的东西。问题出在训练时的反向传播:

  - 训练靠梯度更新参数。梯度要从最后一层,一层层往回传到第一层。
  - 每穿过一层,梯度会乘以那一层的一个数(导数)。
  - 如果这些数普遍 < 1(很常见),梯度就像 0.8 × 0.8 × 0.8 × ...
  连乘,越乘越小,传到浅层时几乎变成 0。
  - 结果:浅层收不到有效梯度 → 学不动 → 整个深网络训练失败。

  这就是 梯度消失(vanishing gradient)。你现在 3 层 Block
  表现变差,就是这个的初级症状。

  残差怎么解决:给梯度修一条"高速公路"

  改成 x = x + layer(x) 后,数据流变成:

       ┌──────────────(直接加过去)──────────────┐
  x ───┤                                          ▼
       └──→ layer(x) ──────────────────────────→ (+) ──→ 输出

  x 有两条路:一条穿过 layer(被加工),一条直接跳过 layer 加到输出(这条叫 skip
  connection / 跳连)。

  关键在反向传播时这条"直接路"的作用:

  x + layer(x) 对 x 求导 = 1 + layer的导数。注意那个 1:

  - 就算 layer的导数 很小(接近 0),梯度里还有个 +1 保底。
  - 所以梯度穿过这一层时,至少是 1 倍(不衰减),而不是被乘成很小。
  - 一路上每层都有这个 +1 的"直通",梯度就能几乎无损地流回最浅层 →
  深层网络也能训练。

  比喻:没有残差,梯度像走一条要翻 10
  座山的路,翻着翻着没力气了(消失)。残差给它修了一条平地高速公路,直接开回起点。

  从"学什么"的角度再理解一遍(残差的名字由来)

  - 没有残差:layer 要学"完整的正确输出"。比如输入是 x,想要输出 H(x),layer
  得从零学出整个 H(x)。难。
  - 有残差:输出 = x + layer(x),想要 H(x),那么 layer(x) 只需学 H(x) -
  x,也就是"输出和输入的差"(residual = 残差、差值)。

  为什么学"差"更容易:
  - 深层网络里,理想的每一层往往只需对输入做微小调整(输出和输入差不多)。
  - 学"微调"(差值接近 0)比学"完整映射"容易得多。
  - 极端情况:如果这一层其实没必要,layer 只要学会输出 0,x + 0 = x 原样传下去 ——
  多余的层不会帮倒忙。没残差的话,让一层学"原样复制输入"反而很难。

  回到你的 Block

  你现在:
  def forward(self, x):
      x = self.sa(x)      # attention 完全替换 x
      x = self.ffwd(x)    # ffwd 又完全替换
      return x
  叠 3 个这样的 Block = 6 次完全替换,梯度传不回去 → 深了反而差。

  改成残差:
  def forward(self, x):
      x = x + self.sa(x)      # attention 只学"要给 x 加什么"
      x = x + self.ffwd(x)    # ffwd 只学"要给 x 加什么"
      return x
  - self.sa(x) 学"该往 x 里加多少上下文信息"。
  - self.ffwd(x) 学"该往 x 里加多少加工后的特征"。
  - 原始 x 一路通过 + 畅通无阻,梯度也能顺着 + 流回去。

  就加两个 x +,深层网络立刻能训练了。

  一个小澄清:形状必须一致才能加

  x + layer(x) 要求 layer(x) 和 x 形状相同(都是 (B,T,32)),才能逐元素相加。你的
  attention 和 ffwd 输出正好都是 (B,T,n_embd),和输入同形,所以能直接加
  ✅。(这也是为什么之前强调"attention 输出维度不变"很重要。)

  一句话总结

  - 问题:网络一深,梯度反向传播时被层层"乘小"→ 消失 → 浅层学不动 → 深了反而差。
  - 残差连接 = x = x + layer(x),让原始输入直接跳过该层加到输出。
  - 两个好处:
  ① 反向传播时那个 +1 给梯度一条不衰减的"高速公路";
  ② 每层只需学"微调(残差)"而非完整映射,更容易学。
  - 代价:几乎为零 —— 就加个 x +,不增加参数。
  - 这是深度网络(ResNet、Transformer)能叠很多层的根本前提 🐾 """

In [21]:
class BatchNorm1d:
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps                   
        self.momentum = momentum         
        self.training = True             
        
        #parameter
        self.gamma = torch.ones(dim)            
        self.beta = torch.zeros(dim)            
        
        #buffer
        self.running_mean = torch.zeros(dim)    
        self.running_var = torch.ones(dim)      

    def __call__(self, x):
        if self.training:
            xmean = x.mean(0, keepdim=True)              
            xvar = x.var(0, keepdim=True)               
        else:
            xmean = self.running_mean                   
            xvar = self.running_var
        
        
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        
        self.out = self.gamma * xhat + self.beta
        
        
        if self.training:
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
                self.running_var  = (1 - self.momentum) * self.running_var  + self.momentum * xvar
        return self.out

    def parameters(self):
        return [self.gamma, self.beta]

In [22]:
class LayerNorm1d:
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps                   
        #parameter
        self.gamma = torch.ones(dim)            
        self.beta = torch.zeros(dim)            
        
    def __call__(self, x):
        
        xmean = x.mean(1, keepdim=True)               
        xvar = x.var(1, keepdim=True)                       
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta
        
        return self.out

    def parameters(self):
        return [self.gamma, self.beta]

In [23]:
torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100)
x = module(x)
x.shape

torch.Size([32, 100])

In [28]:
x[0].mean(), x[0].std()

(tensor(-9.5367e-09), tensor(1.0000))

In [ ]:
""" nn.LayerNorm(n_embd)   # 内部有:
  #   weight (γ, gamma):缩放,形状 (n_embd,),初始全 1
  #   bias   (β, beta) :平移,形状 (n_embd,),初始全 0 """

In [29]:
class Block(nn.Module):
    """ Transformer block: communication followed by computation """
    
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        
    def forward(self, x):        
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x 
        

In [34]:
        
class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd) #number of embedding dimensions
        self.position_embedding_table = nn.Embedding(block_size, n_embd) # each position has its own embd vec
        
        self.blocks = nn.Sequential(
            Block(n_embd, n_head=4), #32//4 = 8 4 attention with 8-dim then computation through ffwd
            Block(n_embd, n_head=4),
            Block(n_embd, n_head=4),  
            nn.LayerNorm(n_embd)
        )
        
        self.lm_head = nn.Linear(n_embd, vocab_size)
        
    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx) #(BTC) C = n_embd
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) #(TC)
        x = tok_emb + pos_emb #now the token holds token identities and position it occur                

        x = self.blocks(x) #BT C = n_embd
    
        logits = self.lm_head(x) #BTC C = vocab_size
        
        
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        #idx is BT
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:] #BT = 8, since block size= 8 in the model, if don trunk the self(idx) cannot work
            logits, loss = self(idx_cond) #BTC
            logits = logits[:, -1, :] #BC
            probs = F.softmax(logits, dim=-1) #BC
            idx_next = torch.multinomial(probs, num_samples = 1) #B1
            idx = torch.cat([idx, idx_next], dim=1) #B T+1
        return idx
    

In [35]:
model = BigramLanguageModel()    
m = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

for iter in range(max_iter):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        
    xb, yb = get_batch('train')        
    
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    

step 0: train loss 4.2698, val loss 4.2719
step 300: train loss 2.5324, val loss 2.5244
step 600: train loss 2.3738, val loss 2.3721
step 900: train loss 2.2602, val loss 2.2889
step 1200: train loss 2.2328, val loss 2.2298
step 1500: train loss 2.1737, val loss 2.1948
step 1800: train loss 2.1405, val loss 2.1783
step 2100: train loss 2.1168, val loss 2.1601
step 2400: train loss 2.0975, val loss 2.1574
step 2700: train loss 2.0769, val loss 2.1235
step 3000: train loss 2.0465, val loss 2.1033
step 3300: train loss 2.0560, val loss 2.1172
step 3600: train loss 2.0358, val loss 2.1034
step 3900: train loss 2.0238, val loss 2.0786
step 4200: train loss 2.0065, val loss 2.0819
step 4500: train loss 1.9986, val loss 2.0845
step 4800: train loss 1.9857, val loss 2.0932


In [38]:
context = torch.zeros((1,1), dtype = torch.long, device = device)
print(decode(m.generate(context, 500)[0].tolist()))


ING HESTINCE:
With
This bargair
Richany Manderst suchs brucesfired quascearn you mopforn.

INGRK:
A:
Hent somes
That the can'll; lidmith postroue
By shang: housurs.

POLOUCIO:
The laike;
Towehs hert what endy, your weind shalst on whath youn the arme no ight geveng farth canturise.

NAUMINVINRIO:
Whow mydaicge that seet me to a is oun the men tistre courses; Make is arbeith wat, forcinglicke.
Now neaty him, but to deping for his we!
Not is;
I and diatwasd;
And tuse
Lick. Why cansers, pach, to ha
